## Scaling

In [1]:
import jax, jax.numpy as jnp

from untangle import algorithm 
from untangle.utils import collect_information, function_error

key = jax.random.key(0)

### Similar Outputs

In [2]:
def f1(x):
    x1, x2 = x
    return jnp.array([
         jnp.cos(4 * jnp.pi * x1 + x2) + x1**2,
        -jnp.sin(3 * jnp.pi * x2)      + 3*x2**2,
    ])

m = 2
n = 2
rank = 2

In [3]:
num_points = 30
X1, Y1, J1 = collect_information(f1, num_points, m, key=key)

In [4]:
learned_f1 = algorithm.cmtf_bsd(X1, Y1, J1, rank, key=key)

Computing CMTF-BSD: 100%|█████████████████████████████████████████████████| 50/50 [00:07<00:00,  7.04it/s]


In [5]:
errors = function_error(f1, learned_f1, X1)
print(f'error(y1) = {errors[0]:.1f}%, error(y2) = {errors[1]:.1f}%')

error(y1) = 3.1%, error(y2) = 2.1%


### Different Magnitudes

In [6]:
def f2(x):
    x1, x2 = x
    return jnp.array([
         100*(jnp.cos(4 * jnp.pi * x1 + x2) + x1**2),
        -jnp.sin(3 * jnp.pi * x2) + 3*x2**2,
    ])

In [7]:
num_points = 30
X2, Y2, J2 = collect_information(f2, num_points, m, key=key)

In [8]:
learned_f2 = algorithm.cmtf_bsd(X2, Y2, J2, rank, key=key)

Computing CMTF-BSD: 100%|█████████████████████████████████████████████████| 50/50 [00:04<00:00, 10.23it/s]


In [9]:
errors = function_error(f2, learned_f2, X2)
print(f'error(y1) = {errors[0]:.0f}%, error(y2) = {errors[1]:.0f}%')

error(y1) = 2%, error(y2) = 100%


### Tensor Scaling

In [10]:
from untangle.scaler import JacobianScaler

In [11]:
scaler = JacobianScaler(J2, Y2)
J2_scaled, Y2_scaled = scaler.scale()

jnp.linalg.norm(J2_scaled[0, :, :]), jnp.linalg.norm(J2_scaled[1, :, :])

(Array(7.7459674, dtype=float32), Array(7.745967, dtype=float32))

In [12]:
learned_f2_scaled = algorithm.cmtf_bsd(X2, Y2_scaled, J2_scaled, rank, key=key)
learned_f2 = scaler.unscale(learned_f2_scaled)

Computing CMTF-BSD: 100%|█████████████████████████████████████████████████| 50/50 [00:04<00:00, 10.25it/s]


In [13]:
errors = function_error(f2, learned_f2, X2)
print(f'error(y1) = {errors[0]:.1f}%, error(y2) = {errors[1]:.1f}%')

error(y1) = 3.1%, error(y2) = 2.1%
